# Wave Bulk Parameter Comparison: SCHISM+WWM vs SCHISM+WW3 (UFS)

This notebook compares wave bulk parameters (Significant Wave Height Hs, Mean Wave Period TM01, and Mean Wave Direction MWD) 
from SCHISM+WWM and WW3 (UFS) runs against NDBC buoy observations.

## Run Configuration Summary

| Run ID | Model | Spec. Boundary | Wind Forcing | Coupling |
|--------|-------|---------------|--------------|----------|
| R09a   | WWM   | Off           | ERA5         | None (Uncoupled) |
| R09b   | WWM   | Off           | ERA5         | 2D Coupled |
| R10a   | WWM   | On            | ERA5         | None (Uncoupled) |
| R10b   | WWM   | On            | ERA5         | 2D Coupled |

WW3 (UFS) runs mirror the same configurations (no spec boundary / with spec boundary, uncoupled / coupled).

## Comparison Groups
- **Uncoupled, No Spec Boundary**: WWM R09a vs WW3 R09a
- **Uncoupled, With Spec Boundary**: WWM R10a vs WW3 R10a  
- **2D Coupled, No Spec Boundary**: WWM R09b vs WW3 R09b
- **2D Coupled, With Spec Boundary**: WWM R10b vs WW3 R10b

## Output
- 3 time-series figures (one per variable), each showing all 8 model runs at all 7 buoy stations
- 3 Taylor diagrams (one per variable), aggregating skill across all stations

## 1. Imports

In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.projections import PolarAxes
import mpl_toolkits.axisartist.grid_finder as gf
import mpl_toolkits.axisartist.floating_axes as fa
from sklearn.metrics import mean_squared_error, r2_score

## 2. Station Metadata & Paths

Update `obs_path`, `wwm_base_path`, `ww3_base_path`, and `output_dir` to match your local file system.

In [ ]:
# ============================================================
# STATION METADATA
# Only NDBC buoys (indices 9–15 in the original station list)
# are used here because they provide spectral wave data.
# ============================================================
NDBC_STATIONS = {
    'codes':  [46035, 46070, 46071, 46072, 46073, 46075, 46265],
    'names':  [
        'Central Bering Sea',
        'SW Bering Sea',
        'Western Aleutians',
        'Central Aleutians',
        'SE Bering Sea',
        'Shumagin Islands',
        'Nome'
    ],
    # Station index in the original 17-station array (0-based)
    'orig_idx': list(range(9, 16)),
    # Station number used in WW3 output file naming (1-based)
    'ww3_num':  list(range(10, 17)),
}

# ============================================================
# PATHS  — update these to match your file system
# ============================================================
obs_path      = r"C:\Users\Felicio.Cassalho\Work\Modeling\AK_Project\WaveCu_paper\buoy_data\spec/"
wwm_base_path = r"C:\Users\Felicio.Cassalho\Work\Modeling\AK_Project\WaveCu_paper"
ww3_base_path = r"C:\Users\Felicio.Cassalho\Work\Modeling\AK_Project\UFS\Waves\SA\WW3"
output_dir    = r"C:\Users\Felicio.Cassalho\Work\Modeling\AK_Project\UFS\Waves\SA\Figures\Bulk_Stations"

# ============================================================
# ANALYSIS PERIOD
# ============================================================
START_DATE = '2019-08-01'
END_DATE   = '2019-10-31'

os.makedirs(output_dir, exist_ok=True)
obs_files = sorted([os.path.join(obs_path, f) for f in os.listdir(obs_path)])

## 3. Model Run Definitions

Each entry in `MODEL_RUNS` defines:
- `label`: human-readable legend name
- `color` / `linestyle`: plot style
- `wwm_run`: subdirectory name under `wwm_base_path` for WWM station files
- `ww3_run`: subdirectory name under `ww3_base_path` for WW3 spectral files

In [ ]:
# ============================================================
# MODEL RUN DEFINITIONS
# ============================================================
# Colors: blues for WWM runs, reds/oranges for WW3 runs
# Solid lines: no spectral boundary; dashed: with spectral boundary
# Lighter shade: uncoupled; darker shade: coupled

MODEL_RUNS = [
    # --- WWM runs ---
    {
        'label':      'WWM – Uncoupled, No Spec. Bnd. (ERA5)',
        'model_type': 'wwm',
        'run_dir':    'R09a',
        'color':      '#3399FF',   # light blue
        'linestyle':  '-',
        'linewidth':  1.4,
    },
    {
        'label':      'WWM – 2D Coupled, No Spec. Bnd. (ERA5)',
        'model_type': 'wwm',
        'run_dir':    'R09b',
        'color':      '#0055CC',   # medium blue
        'linestyle':  '-',
        'linewidth':  1.4,
    },
    {
        'label':      'WWM – Uncoupled, With Spec. Bnd. (ERA5)',
        'model_type': 'wwm',
        'run_dir':    'R10a',
        'color':      '#99CCFF',   # sky blue
        'linestyle':  '--',
        'linewidth':  1.4,
    },
    {
        'label':      'WWM – 2D Coupled, With Spec. Bnd. (ERA5)',
        'model_type': 'wwm',
        'run_dir':    'R10b',
        'color':      '#003399',   # dark blue
        'linestyle':  '--',
        'linewidth':  1.4,
    },
    # --- WW3 (UFS) runs ---
    {
        'label':      'WW3 – Uncoupled, No Spec. Bnd.',
        'model_type': 'ww3',
        'run_dir':    'R09a',
        'color':      '#FF6633',   # light orange
        'linestyle':  '-',
        'linewidth':  1.4,
    },
    {
        'label':      'WW3 – 2D Coupled, No Spec. Bnd.',
        'model_type': 'ww3',
        'run_dir':    'R09b',
        'color':      '#CC2200',   # medium red
        'linestyle':  '-',
        'linewidth':  1.4,
    },
    {
        'label':      'WW3 – Uncoupled, With Spec. Bnd.',
        'model_type': 'ww3',
        'run_dir':    'R10a',
        'color':      '#FFAA88',   # peach
        'linestyle':  '--',
        'linewidth':  1.4,
    },
    {
        'label':      'WW3 – 2D Coupled, With Spec. Bnd.',
        'model_type': 'ww3',
        'run_dir':    'R10b',
        'color':      '#880000',   # dark red
        'linestyle':  '--',
        'linewidth':  1.4,
    },
]

## 4. Helper Functions

In [ ]:
# ============================================================
# DATA CLEANING & STATISTICS
# ============================================================

def clean_da(da):
    """Squeeze, compute, standardise datetime precision, drop duplicates, sort."""
    da = da.squeeze().compute()
    if 'datetime' in da.coords:
        da = da.assign_coords(datetime=da.datetime.astype('datetime64[ns]'))
        _, uniq = np.unique(da['datetime'].values, return_index=True)
        da = da.isel(datetime=uniq).sortby('datetime')
    return da


def calc_stats(obs_da, model_da):
    """Return (RMSE, bias, R²) for scalar variables."""
    obs_da   = clean_da(obs_da)
    model_da = clean_da(model_da)
    try:
        m_interp = model_da.interp(datetime=obs_da.datetime, method='linear')
    except Exception:
        return np.nan, np.nan, np.nan
    valid = np.isfinite(obs_da) & np.isfinite(m_interp)
    o = obs_da.where(valid, drop=True).values
    m = m_interp.where(valid, drop=True).values
    if len(o) == 0:
        return np.nan, np.nan, np.nan
    rmse = np.sqrt(mean_squared_error(o, m))
    bias = float(np.mean(m - o))
    r2   = float(r2_score(o, m))
    return rmse, bias, r2


def calc_circ_stats(obs_da, model_da):
    """Return (circular RMSE, circular bias) for directional variables."""
    obs_da   = clean_da(obs_da)
    model_da = clean_da(model_da)
    try:
        m_interp = model_da.interp(datetime=obs_da.datetime, method='linear')
    except Exception:
        return np.nan, np.nan
    valid = np.isfinite(obs_da) & np.isfinite(m_interp)
    o_deg = obs_da.where(valid, drop=True).values
    m_deg = m_interp.where(valid, drop=True).values
    if len(o_deg) == 0:
        return np.nan, np.nan
    diff_rad = np.arctan2(
        np.sin(np.deg2rad(m_deg) - np.deg2rad(o_deg)),
        np.cos(np.deg2rad(m_deg) - np.deg2rad(o_deg))
    )
    diff_deg = np.rad2deg(diff_rad)
    return float(np.sqrt(np.mean(diff_deg**2))), float(np.mean(diff_deg))


def calc_taylor_stats(obs_da, model_da):
    """Return (std_model_norm, correlation, rmse_norm) normalised by obs std."""
    obs_da   = clean_da(obs_da)
    model_da = clean_da(model_da)
    try:
        m_interp = model_da.interp(datetime=obs_da.datetime, method='linear')
    except Exception:
        return np.nan, np.nan, np.nan
    valid = np.isfinite(obs_da) & np.isfinite(m_interp)
    o = obs_da.where(valid, drop=True).values
    m = m_interp.where(valid, drop=True).values
    if len(o) < 3:
        return np.nan, np.nan, np.nan
    std_obs   = float(np.std(o, ddof=1))
    std_model = float(np.std(m, ddof=1))
    corr      = float(np.corrcoef(o, m)[0, 1])
    rmse_c    = float(np.sqrt(np.mean((m - m.mean() - (o - o.mean()))**2)))
    if std_obs == 0:
        return np.nan, np.nan, np.nan
    return std_model / std_obs, corr, rmse_c / std_obs


def mask_jumps(da, threshold_deg=180):
    """Mask wrap-around discontinuities in directional data."""
    diffs = da.diff(dim='datetime')
    jump  = diffs**2 > threshold_deg**2
    mask_bef = jump.reindex_like(da, fill_value=False)
    mask_aft = jump.shift(datetime=-1, fill_value=False).reindex_like(da, fill_value=False)
    mask = mask_bef | mask_aft
    return da.where(~mask.fillna(False))

## 5. Data Loading

Load all WWM station files and calculate WW3 bulk parameters from spectral output.

In [ ]:
# ============================================================
# LOAD ALL MODEL DATA
# Stored in nested dicts:
#   wwm_data[run_dir][orig_idx]  -> xr.Dataset
#   ww3_data[run_dir][orig_idx]  -> dict with keys 'hs', 'tm01', 'mwd'
# ============================================================

wwm_data = {}
ww3_data = {}

wwm_runs = {r['run_dir'] for r in MODEL_RUNS if r['model_type'] == 'wwm'}
ww3_runs = {r['run_dir'] for r in MODEL_RUNS if r['model_type'] == 'ww3'}

# --- Load WWM runs ---
for run in sorted(wwm_runs):
    run_path = os.path.join(wwm_base_path, run, 'wwm_stations')
    files = sorted([os.path.join(run_path, f) for f in os.listdir(run_path)])
    ds = xr.concat([xr.open_dataset(f) for f in files], dim='ocean_time')
    _, idx = np.unique(ds['ocean_time'], return_index=True)
    wwm_data[run] = ds.isel(ocean_time=idx).sortby('ocean_time')
    print(f'  WWM {run}: loaded {len(files)} files')

# --- Load WW3 runs and compute bulk parameters ---
for run in sorted(ww3_runs):
    run_path = os.path.join(ww3_base_path, run)
    ww3_data[run] = {}
    for i, (orig_idx, ww3_num) in enumerate(
            zip(NDBC_STATIONS['orig_idx'], NDBC_STATIONS['ww3_num'])):
        ww3_file = os.path.join(run_path, f'station_ww3.{ww3_num}_2019_spec.nc')
        ds_ww3 = xr.open_dataset(ww3_file).squeeze().sortby('direction')

        dir_rad = np.deg2rad(ds_ww3['direction'])
        Ef = ds_ww3['efth'].integrate('direction') * (np.pi / 180.0)

        m0 = Ef.integrate('frequency')
        m1 = (Ef * ds_ww3['frequency']).integrate('frequency')

        hs   = 4 * np.sqrt(m0.clip(min=0))
        tm01 = m0 / m1

        m_sin = (ds_ww3['efth'] * np.sin(dir_rad)).integrate('direction').integrate('frequency')
        m_cos = (ds_ww3['efth'] * np.cos(dir_rad)).integrate('direction').integrate('frequency')
        mwd   = (np.rad2deg(np.arctan2(m_sin, m_cos)) + 180) % 360   # convert 'going-to' → 'coming-from'

        ww3_data[run][orig_idx] = {'hs': hs, 'tm01': tm01, 'mwd': mwd}
    print(f'  WW3  {run}: bulk parameters computed for {i+1} stations')

# --- Pre-load observation spectral files (one per NDBC station) ---
# obs_files is sorted; they correspond to the 7 NDBC stations in order.
print('\nData loading complete.')

## 6. Figure 1 – Significant Wave Height (Hs)

All 8 model runs overlaid on observations at each of the 7 NDBC buoy stations.

In [ ]:
# ============================================================
# FIGURE 1: SIGNIFICANT WAVE HEIGHT (Hs)
# ============================================================
n_stations = len(NDBC_STATIONS['codes'])
fig, axes = plt.subplots(n_stations, 1, figsize=(20, 3.2 * n_stations), sharex=False)

for i, (orig_idx, code, name) in enumerate(
        zip(NDBC_STATIONS['orig_idx'],
            NDBC_STATIONS['codes'],
            NDBC_STATIONS['names'])):

    ax = axes[i]

    # --- Observation ---
    obs_ds = xr.open_dataset(obs_files[i]).squeeze()
    m0_obs = obs_ds['spectral_wave_density'].integrate('frequency')
    obs_hs = (4 * np.sqrt(m0_obs)).rename({'time': 'datetime'})
    obs_sel = clean_da(obs_hs.sel(datetime=slice(START_DATE, END_DATE)))

    ax.plot(obs_sel['datetime'], obs_sel,
            color='black', linewidth=1.8, label='Observation', zorder=10)

    # --- Model runs ---
    for run_cfg in MODEL_RUNS:
        run  = run_cfg['run_dir']
        mtype = run_cfg['model_type']

        if mtype == 'wwm':
            da = wwm_data[run]['HS'][:, orig_idx].rename({'ocean_time': 'datetime'})
        else:
            da = ww3_data[run][orig_idx]['hs'].rename({'time': 'datetime'})

        da_sel = clean_da(da.sel(datetime=slice(START_DATE, END_DATE)))
        rmse, bias, r2 = calc_stats(obs_sel, da_sel)
        r2_str = f'{r2:.2f}' if np.isfinite(r2) and r2 >= 0 else '<0'
        lbl = (f"{run_cfg['label']:<50}  "
               f"RMSE={rmse:.2f} m  Bias={bias:+.2f} m  R²={r2_str}")

        ax.plot(da_sel['datetime'], da_sel,
                color=run_cfg['color'],
                linestyle=run_cfg['linestyle'],
                linewidth=run_cfg['linewidth'],
                label=lbl)

    # --- Formatting ---
    ax.set_ylabel(f'Hs (m)\nNDBC {code}\n{name}', fontsize=11)
    ax.set_ylim(0, 14)
    ax.set_xlim(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))
    ax.grid(True, linestyle='--', color='gray', linewidth=0.4, alpha=0.7)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
              borderaxespad=0., fontsize=9, frameon=False)

    if i == n_stations - 1:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.setp(ax.get_xticklabels(), fontsize=11)
    else:
        ax.tick_params(axis='x', labelbottom=False)

fig.suptitle(
    'Significant Wave Height (Hs): SCHISM+WWM vs WW3(UFS) vs Observations\n'
    f'Period: {START_DATE} – {END_DATE}',
    fontsize=13, y=1.002
)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, 'Hs_all_runs_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

## 7. Figure 2 – Mean Wave Period (TM01)

In [ ]:
# ============================================================
# FIGURE 2: MEAN WAVE PERIOD (TM01)
# ============================================================
fig, axes = plt.subplots(n_stations, 1, figsize=(20, 3.2 * n_stations), sharex=False)

for i, (orig_idx, code, name) in enumerate(
        zip(NDBC_STATIONS['orig_idx'],
            NDBC_STATIONS['codes'],
            NDBC_STATIONS['names'])):

    ax = axes[i]

    # --- Observation ---
    obs_ds  = xr.open_dataset(obs_files[i]).squeeze()
    E       = obs_ds['spectral_wave_density']
    f       = obs_ds['frequency']
    m0_obs  = E.integrate('frequency')
    m1_obs  = (E * f).integrate('frequency')
    obs_tm  = (m0_obs / m1_obs).rename({'time': 'datetime'})
    obs_sel = clean_da(obs_tm.sel(datetime=slice(START_DATE, END_DATE)))

    ax.plot(obs_sel['datetime'], obs_sel,
            color='black', linewidth=1.8, label='Observation', zorder=10)

    # --- Model runs ---
    for run_cfg in MODEL_RUNS:
        run   = run_cfg['run_dir']
        mtype = run_cfg['model_type']

        if mtype == 'wwm':
            da = wwm_data[run]['TM01'][:, orig_idx].rename({'ocean_time': 'datetime'})
        else:
            da = ww3_data[run][orig_idx]['tm01'].rename({'time': 'datetime'})

        da_sel = clean_da(da.sel(datetime=slice(START_DATE, END_DATE)))
        rmse, bias, r2 = calc_stats(obs_sel, da_sel)
        r2_str = f'{r2:.2f}' if np.isfinite(r2) and r2 >= 0 else '<0'
        lbl = (f"{run_cfg['label']:<50}  "
               f"RMSE={rmse:.2f} s  Bias={bias:+.2f} s  R²={r2_str}")

        ax.plot(da_sel['datetime'], da_sel,
                color=run_cfg['color'],
                linestyle=run_cfg['linestyle'],
                linewidth=run_cfg['linewidth'],
                label=lbl)

    # --- Formatting ---
    ax.set_ylabel(f'TM01 (s)\nNDBC {code}\n{name}', fontsize=11)
    ax.set_ylim(2, 13)
    ax.set_xlim(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))
    ax.grid(True, linestyle='--', color='gray', linewidth=0.4, alpha=0.7)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
              borderaxespad=0., fontsize=9, frameon=False)

    if i == n_stations - 1:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.setp(ax.get_xticklabels(), fontsize=11)
    else:
        ax.tick_params(axis='x', labelbottom=False)

fig.suptitle(
    'Mean Wave Period (TM01): SCHISM+WWM vs WW3(UFS) vs Observations\n'
    f'Period: {START_DATE} – {END_DATE}',
    fontsize=13, y=1.002
)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, 'TM01_all_runs_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

## 8. Figure 3 – Mean Wave Direction (MWD)

In [ ]:
# ============================================================
# FIGURE 3: MEAN WAVE DIRECTION (MWD)
# ============================================================
fig, axes = plt.subplots(n_stations, 1, figsize=(20, 3.2 * n_stations), sharex=False)

for i, (orig_idx, code, name) in enumerate(
        zip(NDBC_STATIONS['orig_idx'],
            NDBC_STATIONS['codes'],
            NDBC_STATIONS['names'])):

    ax = axes[i]

    # --- Observation ---
    obs_ds   = xr.open_dataset(obs_files[i]).squeeze()
    E        = obs_ds['spectral_wave_density']
    theta_rad = np.deg2rad(obs_ds['mean_wave_dir'])
    m_sin_obs = (E * np.sin(theta_rad)).integrate('frequency')
    m_cos_obs = (E * np.cos(theta_rad)).integrate('frequency')
    obs_mwd  = (np.rad2deg(np.arctan2(m_sin_obs, m_cos_obs)) % 360).rename({'time': 'datetime'})
    obs_sel  = clean_da(obs_mwd.sel(datetime=slice(START_DATE, END_DATE)))

    ax.scatter(obs_sel['datetime'], obs_sel,
               color='black', marker='x', s=8, label='Observation', zorder=10)

    # --- Model runs ---
    for run_cfg in MODEL_RUNS:
        run   = run_cfg['run_dir']
        mtype = run_cfg['model_type']

        if mtype == 'wwm':
            da = wwm_data[run]['DM'][:, orig_idx].rename({'ocean_time': 'datetime'})
        else:
            da = ww3_data[run][orig_idx]['mwd'].rename({'time': 'datetime'})

        da_sel  = clean_da(da.sel(datetime=slice(START_DATE, END_DATE)))
        rmse, bias = calc_circ_stats(obs_sel, da_sel)
        da_plot = mask_jumps(da_sel)
        lbl = (f"{run_cfg['label']:<50}  "
               f"RMSE={rmse:.1f}°  Bias={bias:+.1f}°")

        ax.plot(da_plot['datetime'], da_plot,
                color=run_cfg['color'],
                linestyle=run_cfg['linestyle'],
                linewidth=run_cfg['linewidth'],
                label=lbl)

    # --- Formatting ---
    ax.set_ylabel(f'Direction (°)\nNDBC {code}\n{name}', fontsize=11)
    ax.set_yticks([0, 90, 180, 270, 360])
    ax.set_ylim(0, 360)
    ax.set_xlim(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))
    ax.grid(True, linestyle='--', color='gray', linewidth=0.4, alpha=0.7)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
              borderaxespad=0., fontsize=9, frameon=False)

    if i == n_stations - 1:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.setp(ax.get_xticklabels(), fontsize=11)
    else:
        ax.tick_params(axis='x', labelbottom=False)

fig.suptitle(
    'Mean Wave Direction (MWD): SCHISM+WWM vs WW3(UFS) vs Observations\n'
    f'Period: {START_DATE} – {END_DATE}',
    fontsize=13, y=1.002
)
plt.tight_layout()
fig.savefig(os.path.join(output_dir, 'MWD_all_runs_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

## 9. Taylor Diagram Class

In [ ]:
# ============================================================
# TAYLOR DIAGRAM CLASS
# Based on the implementation in AK_paper/taylor_diagram.ipynb.
# Points are plotted in normalised space (σ_model / σ_obs).
# ============================================================

class TaylorDiagram:
    """Taylor diagram using floating polar axes."""

    def __init__(self, fig=None, rect=111, label='Reference'):
        self.fig = fig or plt.figure(figsize=(9, 7))
        tr = PolarAxes.PolarTransform()

        # Correlation tick locations
        rlocs  = np.concatenate([np.arange(0, 1.1, 0.1), [0.95, 0.99]])
        tlocs  = np.arccos(rlocs)
        gl1    = gf.FixedLocator(tlocs)
        tf1    = gf.DictFormatter(dict(zip(tlocs, [f'{r:.2f}' for r in rlocs])))

        smin, smax = 0, 2.0   # normalised std range
        gh = fa.GridHelperCurveLinear(
            tr,
            extremes=(0, np.pi / 2, smin, smax),
            grid_locator1=gl1,
            tick_formatter1=tf1,
        )
        ax = fa.FloatingSubplot(self.fig, rect, grid_helper=gh)
        self.fig.add_subplot(ax)

        ax.axis['top'].set_axis_direction('bottom')
        ax.axis['top'].label.set_text('Correlation Coefficient')
        ax.axis['top'].toggle(ticklabels=True, label=True)
        ax.axis['top'].major_ticklabels.set_axis_direction('top')
        ax.axis['top'].label.set_axis_direction('top')

        ax.axis['left'].set_axis_direction('bottom')
        ax.axis['left'].label.set_text('Normalised Std. Dev.')
        ax.axis['left'].toggle(ticklabels=True, label=True)

        ax.axis['right'].set_axis_direction('top')
        ax.axis['right'].label.set_text('Normalised Std. Dev.')
        ax.axis['right'].toggle(ticklabels=True, label=True)
        ax.axis['right'].major_ticklabels.set_axis_direction('left')

        ax.axis['bottom'].set_visible(False)
        ax.grid(True, alpha=0.4)

        self._ax = ax
        self.ax  = ax.get_aux_axes(tr)

        # Reference point at (std=1, corr=1) → angle=0
        self.ax.plot([0], [1], 'k*', ms=12, label=label, zorder=5)

        # Reference std circle
        t = np.linspace(0, np.pi / 2)
        self.ax.plot(t, np.ones_like(t), 'k--', linewidth=0.8, label='_')

        # RMSE contours
        rs, ts = np.meshgrid(np.linspace(smin, smax, 200),
                             np.linspace(0, np.pi / 2, 200))
        rms = np.sqrt(1 + rs**2 - 2 * rs * np.cos(ts))
        cs  = self.ax.contour(ts, rs, rms, levels=[0.5, 1.0, 1.5],
                              colors='green', linewidths=0.8, alpha=0.6)
        self.ax.clabel(cs, fmt='%.1f', fontsize=8)

        self.sample_points = []

    def add_sample(self, std_norm, corr, *args, **kwargs):
        """Add a model point. std_norm = σ_model/σ_obs, corr = Pearson r."""
        theta = np.arccos(np.clip(corr, -1, 1))
        l, = self.ax.plot(theta, std_norm, *args, **kwargs)
        self.sample_points.append(l)
        return l

## 10. Aggregate Taylor Statistics

Compute station-averaged Taylor statistics (normalised std dev and correlation) for each model run.

In [ ]:
# ============================================================
# AGGREGATE TAYLOR STATS ACROSS ALL 7 STATIONS
# For each model run, concatenate all observation and model
# data (after time-interpolation) across stations, then
# compute a single (std_norm, corr) pair.
# ============================================================

def agg_taylor_stats(run_cfg, variable='hs'):
    """
    Aggregate Taylor stats across all NDBC stations for the given variable.
    variable: 'hs' | 'tm01' | 'mwd'
    Returns (std_norm, corr) or (np.nan, np.nan).
    """
    all_obs, all_mod = [], []
    run   = run_cfg['run_dir']
    mtype = run_cfg['model_type']

    for i, (orig_idx, obs_file) in enumerate(
            zip(NDBC_STATIONS['orig_idx'], obs_files)):

        # --- Observation ---
        obs_ds = xr.open_dataset(obs_file).squeeze()
        if variable == 'hs':
            m0     = obs_ds['spectral_wave_density'].integrate('frequency')
            obs_da = (4 * np.sqrt(m0)).rename({'time': 'datetime'})
        elif variable == 'tm01':
            E      = obs_ds['spectral_wave_density']
            m0     = E.integrate('frequency')
            m1     = (E * obs_ds['frequency']).integrate('frequency')
            obs_da = (m0 / m1).rename({'time': 'datetime'})
        elif variable == 'mwd':
            E         = obs_ds['spectral_wave_density']
            theta_rad = np.deg2rad(obs_ds['mean_wave_dir'])
            ms        = (E * np.sin(theta_rad)).integrate('frequency')
            mc        = (E * np.cos(theta_rad)).integrate('frequency')
            obs_da    = (np.rad2deg(np.arctan2(ms, mc)) % 360).rename({'time': 'datetime'})
        else:
            raise ValueError(f'Unknown variable: {variable}')

        obs_sel = clean_da(obs_da.sel(datetime=slice(START_DATE, END_DATE)))

        # --- Model ---
        if mtype == 'wwm':
            var_map = {'hs': 'HS', 'tm01': 'TM01', 'mwd': 'DM'}
            da = wwm_data[run][var_map[variable]][:, orig_idx].rename(
                    {'ocean_time': 'datetime'})
        else:
            da = ww3_data[run][orig_idx][variable].rename({'time': 'datetime'})

        da_sel = clean_da(da.sel(datetime=slice(START_DATE, END_DATE)))

        # --- Interpolate model → obs time axis ---
        try:
            m_interp = clean_da(da_sel).interp(datetime=obs_sel.datetime, method='linear')
        except Exception:
            continue

        valid = np.isfinite(obs_sel) & np.isfinite(m_interp)
        all_obs.append(obs_sel.where(valid, drop=True).values)
        all_mod.append(m_interp.where(valid, drop=True).values)

    if not all_obs:
        return np.nan, np.nan

    o = np.concatenate(all_obs)
    m = np.concatenate(all_mod)
    if len(o) < 3:
        return np.nan, np.nan

    std_obs   = np.std(o, ddof=1)
    std_model = np.std(m, ddof=1)
    if std_obs == 0:
        return np.nan, np.nan
    corr = np.corrcoef(o, m)[0, 1]
    return float(std_model / std_obs), float(corr)


# Pre-compute all Taylor stats
taylor_stats = {}
for variable in ('hs', 'tm01', 'mwd'):
    taylor_stats[variable] = []
    for run_cfg in MODEL_RUNS:
        std_n, corr = agg_taylor_stats(run_cfg, variable=variable)
        taylor_stats[variable].append((std_n, corr))
        print(f"  {variable.upper()}  {run_cfg['label'][:45]:<45}  "
              f"std_norm={std_n:.3f}  corr={corr:.3f}")

print('\nTaylor statistics computed.')

## 11. Figure 4 – Taylor Diagram: Significant Wave Height (Hs)

In [ ]:
# ============================================================
# FIGURE 4: TAYLOR DIAGRAM – Hs
# ============================================================
fig = plt.figure(figsize=(10, 8))
td  = TaylorDiagram(fig=fig, rect=111, label='Observation (ref.)')

MARKERS = ['o', 's', '^', 'D', 'o', 's', '^', 'D']

for k, (run_cfg, (std_n, corr)) in enumerate(
        zip(MODEL_RUNS, taylor_stats['hs'])):
    if not (np.isfinite(std_n) and np.isfinite(corr)):
        continue
    td.add_sample(
        std_n, corr,
        marker=MARKERS[k],
        color=run_cfg['color'],
        ms=9,
        ls='',
        markeredgecolor='k',
        markeredgewidth=0.5,
        label=run_cfg['label'],
        zorder=5,
    )

handles = [td._ax.get_aux_axes(PolarAxes.PolarTransform()).lines[0]] if False else []
fig.legend(
    [p for p in td.sample_points] + td._ax.get_aux_axes(PolarAxes.PolarTransform()).lines[:1],
    [p.get_label() for p in td.sample_points] + ['Observation (ref.)'],
    loc='lower right', fontsize=9, frameon=True,
    bbox_to_anchor=(0.98, 0.02),
)
fig.suptitle(
    'Taylor Diagram – Significant Wave Height (Hs)\n'
    'Station-aggregated, normalised by observed std. dev.',
    fontsize=12
)
fig.savefig(os.path.join(output_dir, 'Taylor_Hs.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

## 12. Figure 5 – Taylor Diagram: Mean Wave Period (TM01)

In [ ]:
# ============================================================
# FIGURE 5: TAYLOR DIAGRAM – TM01
# ============================================================
fig = plt.figure(figsize=(10, 8))
td  = TaylorDiagram(fig=fig, rect=111, label='Observation (ref.)')

for k, (run_cfg, (std_n, corr)) in enumerate(
        zip(MODEL_RUNS, taylor_stats['tm01'])):
    if not (np.isfinite(std_n) and np.isfinite(corr)):
        continue
    td.add_sample(
        std_n, corr,
        marker=MARKERS[k],
        color=run_cfg['color'],
        ms=9,
        ls='',
        markeredgecolor='k',
        markeredgewidth=0.5,
        label=run_cfg['label'],
        zorder=5,
    )

fig.legend(
    td.sample_points,
    [p.get_label() for p in td.sample_points],
    loc='lower right', fontsize=9, frameon=True,
    bbox_to_anchor=(0.98, 0.02),
)
fig.suptitle(
    'Taylor Diagram – Mean Wave Period (TM01)\n'
    'Station-aggregated, normalised by observed std. dev.',
    fontsize=12
)
fig.savefig(os.path.join(output_dir, 'Taylor_TM01.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

## 13. Figure 6 – Taylor Diagram: Mean Wave Direction (MWD)

> **Note on direction Taylor diagrams**: The standard Taylor diagram assumes a linear variable.
> Here we treat MWD as a linear variable computed from the Pearson correlation between the
> raw degree values (0–360) after time-interpolation. This is an approximation; for cases
> with strong wrap-around near 0°/360° the correlation may be misleading. Interpret with care.

In [ ]:
# ============================================================
# FIGURE 6: TAYLOR DIAGRAM – MWD
# ============================================================
fig = plt.figure(figsize=(10, 8))
td  = TaylorDiagram(fig=fig, rect=111, label='Observation (ref.)')

for k, (run_cfg, (std_n, corr)) in enumerate(
        zip(MODEL_RUNS, taylor_stats['mwd'])):
    if not (np.isfinite(std_n) and np.isfinite(corr)):
        continue
    # Only plot points with positive correlation (otherwise outside the quarter-circle)
    if corr < 0:
        print(f"  Skipping {run_cfg['label']} (corr={corr:.2f} < 0 – outside diagram)")
        continue
    td.add_sample(
        std_n, corr,
        marker=MARKERS[k],
        color=run_cfg['color'],
        ms=9,
        ls='',
        markeredgecolor='k',
        markeredgewidth=0.5,
        label=run_cfg['label'],
        zorder=5,
    )

fig.legend(
    td.sample_points,
    [p.get_label() for p in td.sample_points],
    loc='lower right', fontsize=9, frameon=True,
    bbox_to_anchor=(0.98, 0.02),
)
fig.suptitle(
    'Taylor Diagram – Mean Wave Direction (MWD)\n'
    'Station-aggregated, normalised by observed std. dev.\n'
    '(Direction treated as linear; see note in markdown cell above)',
    fontsize=11
)
fig.savefig(os.path.join(output_dir, 'Taylor_MWD.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

## 14. Summary Statistics Table

Print a compact table of RMSE and Bias (station-averaged) for each run and variable.

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================
import warnings
warnings.filterwarnings('ignore')

rows = []
for run_cfg in MODEL_RUNS:
    row = {'Run': run_cfg['label']}
    run   = run_cfg['run_dir']
    mtype = run_cfg['model_type']

    for variable, varlabel, var_map_wwm in [
            ('hs',   'Hs',   'HS'),
            ('tm01', 'TM01', 'TM01'),
            ('mwd',  'MWD',  'DM')]:

        rmse_list, bias_list = [], []
        for i, (orig_idx, obs_file) in enumerate(
                zip(NDBC_STATIONS['orig_idx'], obs_files)):

            obs_ds = xr.open_dataset(obs_file).squeeze()
            if variable == 'hs':
                m0  = obs_ds['spectral_wave_density'].integrate('frequency')
                obs = (4 * np.sqrt(m0)).rename({'time': 'datetime'})
            elif variable == 'tm01':
                E  = obs_ds['spectral_wave_density']
                m0 = E.integrate('frequency')
                m1 = (E * obs_ds['frequency']).integrate('frequency')
                obs = (m0 / m1).rename({'time': 'datetime'})
            else:
                E   = obs_ds['spectral_wave_density']
                thr = np.deg2rad(obs_ds['mean_wave_dir'])
                ms  = (E * np.sin(thr)).integrate('frequency')
                mc  = (E * np.cos(thr)).integrate('frequency')
                obs = (np.rad2deg(np.arctan2(ms, mc)) % 360).rename({'time': 'datetime'})

            obs_sel = clean_da(obs.sel(datetime=slice(START_DATE, END_DATE)))

            if mtype == 'wwm':
                da = wwm_data[run][var_map_wwm][:, orig_idx].rename({'ocean_time': 'datetime'})
            else:
                da = ww3_data[run][orig_idx][variable].rename({'time': 'datetime'})
            da_sel = clean_da(da.sel(datetime=slice(START_DATE, END_DATE)))

            if variable == 'mwd':
                r, b = calc_circ_stats(obs_sel, da_sel)
            else:
                r, b, _ = calc_stats(obs_sel, da_sel)

            if np.isfinite(r):
                rmse_list.append(r)
                bias_list.append(b)

        unit = 'm' if variable == 'hs' else ('s' if variable == 'tm01' else '°')
        row[f'{varlabel} RMSE ({unit})'] = f'{np.mean(rmse_list):.3f}' if rmse_list else 'N/A'
        row[f'{varlabel} Bias ({unit})'] = f'{np.mean(bias_list):+.3f}' if bias_list else 'N/A'

    rows.append(row)

df_stats = pd.DataFrame(rows).set_index('Run')
print('\n=== Station-Averaged Statistics ===')
print(df_stats.to_string())